# Farm game art: full painterly run (Colab)

Makes every piece of art in the game (each item at each stage of its lifecycle) from the reference drawings,
in the painterly style, then removes the background, sizes it to the grid and zips it for Claude.

**Steps**
1. Runtime → Change runtime type → **A100** (FLUX 3 Klein), or L4 (FLUX.2 klein 4B).
2. Run all. Cell 3 asks you to upload **`farm_art_refs.zip`** (Claude sends it).
3. Cell 3 also asks to connect Google Drive, so a dropped session loses nothing: run it again and it carries on.
4. At the end your browser downloads **`farm_art_final.zip`**. Attach it in the chat.

About 550 pieces (every crop, tree, animal, building, vehicle and decoration at every stage): roughly 3 hours on an A100.
Pieces already in your Drive are skipped, so a second run only makes what's new.

**Open it from GitHub:** in Colab, File → Open notebook → GitHub, tick *Include private repos*, pick
`aniket-patra-1998/farm-game` and `tools/generate_art.ipynb`. Or go straight to
https://colab.research.google.com/github/aniket-patra-1998/farm-game/blob/main/tools/generate_art.ipynb

In [ ]:
# 1. Settings
STYLE = "painterly"
ONLY = []          # art ids to (re)make, e.g. ["cow", "crop_wheat_ripe"]; empty = everything
SKIP_CLASSES = ["Material"]   # menu icons come later
SEED = 7
REFS_VERSION = 2   # the references zip this notebook expects; a newer one is asked for automatically
BACKEND = "auto"   # "auto", "flux3" (needs ~35 GB GPU, HF_TOKEN), "flux2" (~14 GB GPU) or "api" (BFL_API_KEY)

In [ ]:
# 2. Install
!pip -q install git+https://github.com/huggingface/diffusers.git transformers accelerate sentencepiece protobuf
!pip -q install "rembg[gpu]" pillow requests

In [ ]:
# 3. References and a Drive folder that survives disconnects
import os, json, zipfile
from google.colab import files, drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/farm_art"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
have = json.load(open("manifest.json")).get("version", 1) if os.path.exists("manifest.json") else 0
if have < REFS_VERSION:
    print("Upload farm_art_refs.zip" + (" (the new one: more items were added)" if have else ""))
    up = files.upload()
    zipfile.ZipFile(list(up.keys())[0]).extractall(WORK)   # keeps raw/ and final/: finished pieces aren't redone
manifest = json.load(open("manifest.json"))
assert manifest.get("version", 1) >= REFS_VERSION, "That zip is an old one: use the latest farm_art_refs.zip from Claude"
items = [i for i in manifest["items"] if i["class"] not in SKIP_CLASSES and (not ONLY or i["id"] in ONLY)]
os.makedirs("raw", exist_ok=True)
todo = [i for i in items if not os.path.exists(f"raw/{i['id']}.png")]
print(len(items), "pieces,", len(todo), "still to make")
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
BFL_KEY = secret("BFL_API_KEY")

In [ ]:
# 4. Pick and load the model
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import base64, io, time, requests, torch, gc
from PIL import Image
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if BACKEND == "auto":
    BACKEND = "flux3" if gpu_gb >= 35 and os.environ.get("HF_TOKEN") else ("flux2" if gpu_gb >= 14 else "api")
print(f"GPU: {gpu_gb:.0f} GB -> {BACKEND}")

def refs_for(item):
    # The item's layout reference(s) first, then the approved painterly pieces as style references.
    paths = list(item["refs"]) + list(manifest.get("style_refs", []))
    return [Image.open(r).convert("RGB") for r in paths if os.path.exists(r)]

if BACKEND == "flux3":
    from diffusers import ModularPipeline
    pipe = ModularPipeline.from_pretrained("black-forest-labs/flux-3-image-klein")
    try:
        pipe.blocks.sub_blocks.pop("prompt_upsampler")
        pipe = pipe.blocks.init_pipeline("black-forest-labs/flux-3-image-klein")
    except Exception as e:
        print("Upsampler not removed:", e)
    pipe.load_components(dtype={"vae": torch.float32, "default": torch.bfloat16})
    pipe.to("cuda")
    if getattr(pipe, "vae", None) is not None and hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()

    def generate(prompt, refs, seed):
        args = dict(prompt=prompt, generator=torch.Generator().manual_seed(seed), output="images")
        if refs:
            args["image"] = refs
        for w, h in ((1024, 1024), (768, 768)):
            for extra in ({"height": h, "width": w}, {}):
                try:
                    return pipe(**args, **extra)[0]
                except TypeError:
                    continue
                except torch.OutOfMemoryError:
                    print("out of memory at", w, "- trying smaller")
                    break
                finally:
                    gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError("generation failed")

elif BACKEND == "flux2":
    from diffusers import Flux2KleinPipeline
    pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()

    def generate(prompt, refs, seed):
        args = dict(prompt=prompt, height=1024, width=1024, guidance_scale=1.0, num_inference_steps=4,
                    generator=torch.Generator(device="cuda").manual_seed(seed))
        if refs:
            args["image"] = refs if len(refs) > 1 else refs[0]
        return pipe(**args).images[0]

else:
    assert BFL_KEY, "No GPU and no BFL_API_KEY secret."
    API = "https://api.bfl.ai/v1/flux-3-image-klein"
    HEAD = {"x-key": BFL_KEY, "accept": "application/json", "Content-Type": "application/json"}

    def generate(prompt, refs, seed):
        body = {"prompt": prompt, "seed": seed, "width": 1024, "height": 1024, "output_format": "png"}
        if refs:
            enc = []
            for r in refs:
                b = io.BytesIO(); r.save(b, "PNG"); enc.append(base64.b64encode(b.getvalue()).decode())
            body["images"] = enc
        r = requests.post(API, headers=HEAD, json=body, timeout=60)
        r.raise_for_status()
        poll = r.json()["polling_url"]
        while True:
            time.sleep(0.8)
            res = requests.get(poll, headers=HEAD, timeout=60).json()
            if res.get("status") == "Ready":
                return Image.open(io.BytesIO(requests.get(res["result"]["sample"], timeout=120).content))
            if res.get("status") in ("Error", "Failed", "Content Moderated", "Request Moderated"):
                raise RuntimeError(str(res))
print("ready")

In [ ]:
# 5. Generate everything still to make (saved to Drive as it goes; rerun to resume)
import time as _t
start = _t.time()
failed = []
for k, item in enumerate(todo):
    prompt = f"{manifest['common']} Subject: {item['subject']}. {manifest['styles'][STYLE]}"
    try:
        generate(prompt, refs_for(item), SEED).save(f"raw/{item['id']}.png")
    except Exception as e:
        failed.append(item["id"]); print("FAILED", item["id"], e)
    if k % 10 == 0:
        per = (_t.time() - start) / (k + 1)
        print(f"{k + 1}/{len(todo)}  ~{per * (len(todo) - k - 1) / 60:.0f} min left")
print("done;", len(failed), "failed:", failed)

In [ ]:
# 6. Cut out, size to the grid, record each anchor (the ground centre)
# At 2x a tile is 256 px wide; the footprint's front corner is the image bottom,
# so the ground centre sits 64 px per tile above it.
from rembg import remove, new_session
session = new_session("birefnet-general")
TILE = 256
os.makedirs("final", exist_ok=True)
meta = {}
for item in items:
    src = f"raw/{item['id']}.png"
    if not os.path.exists(src):
        continue
    out = f"final/{item['id']}.png"
    if not os.path.exists(out):
        cut = remove(Image.open(src).convert("RGB"), session=session)
        cut = cut.crop(cut.getbbox())
        w = int(TILE * item["tiles"] * item["fill"])
        cut = cut.resize((w, max(1, int(cut.height * w / cut.width))), Image.LANCZOS)
        cut.save(out)
    im = Image.open(out)
    lift = 64 * item["tiles"] * item["fill"] if item["class"] in ("Crop", "Ground") or item["fill"] >= 0.95 else 0
    meta[item["id"]] = {"anchor": [im.width / 2, im.height - lift], "scale": 0.5}
json.dump(meta, open("final/sprites.json", "w"), indent=1)
print(len(meta), "pieces ready")

In [ ]:
# 7. A quick look
from PIL import ImageDraw
names = sorted(meta)[:48]
cell = 220; cols = 8
sheet = Image.new("RGBA", (cell * cols, cell * ((len(names) + cols - 1) // cols)), (143, 207, 95, 255))
for k, n in enumerate(names):
    im = Image.open(f"final/{n}.png"); im.thumbnail((cell - 20, cell - 20))
    sheet.alpha_composite(im, ((k % cols) * cell + (cell - im.width) // 2, (k // cols) * cell + (cell - im.height) // 2))
display(sheet.convert("RGB"))

In [ ]:
# 8. Zip and download (attach the zip in the chat)
import shutil
shutil.make_archive("/content/farm_art_final", "zip", "final")
files.download("/content/farm_art_final.zip")
print("downloading farm_art_final.zip")